# Jinja Customer Support Intent Detection

To run this notebook, create a virtual environment and install the packages provided in the `requirements.txt`.

### Step 1: Import the necessary packages

In [52]:
from pathlib import Path
import json
from collections import Counter
from jinja2 import Environment, FileSystemLoader, StrictUndefined
from IPython.display import Markdown, display

### Step 2: Create the project structure 

In [53]:
DATA_DIR = Path("data")
TEMPLATES_DIR = Path("templates")
OUTPUTS_DIR = Path("outputs")

for directory in [DATA_DIR, TEMPLATES_DIR, OUTPUTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


### Step3: Create a small dataset of support requests

In [54]:
support_requests = [
    {
        "id": "req001",
        "customer_name": "Max Mustermann",
        "text": "The application crashes every time I try to export CSV files larger than 10MB. I've attached the error log.",
        "product_type": "data_analytics_tool",
        "categories": ["export", "crash", "performance"],
        "urgency_level": "high",
        "contains_sarcasm": False,
        "language": "en",
        "expected_intent": "bug_report",
        "has_attachments": True,
        "os": "Windows 11",
        "additional_info": "This started after the last update."
    },
    {
        "id": "req002",
        "customer_name": "Rainer Zufall",
        "text": "Oh fantastic, the 'Improve User Experience' update broke the entire user experience. Now nothing works.",
        "product_type": "mobile_app",
        "categories": ["ui", "update", "bug"],
        "urgency_level": "critical",
        "contains_sarcasm": True,
        "language": "en",
        "expected_intent": "bug_report",
        "has_attachments": False,
        "os": "iOS 17",
        "additional_info": None
    },
    {
        "id": "req003",
        "customer_name": "Ernst Haft",
        "text": "I need help with API authentication and also with setting up webhook notifications for our enterprise plan.",
        "product_type": "api_service",
        "categories": ["api", "authentication", "webhooks", "enterprise"],
        "urgency_level": "high",
        "contains_sarcasm": False,
        "language": "en",
        "expected_intent": "technical_support",
        "has_attachments": False,
        "os": None,
        "additional_info": "We have a team of 5 developers waiting on this."
    },
    {
        "id": "req004",
        "customer_name": "Anna Nass",
        "text": "The database connection times out after exactly 30 minutes of inactivity. Is there a configurable timeout setting?",
        "product_type": "backend_service",
        "categories": ["database", "timeout", "configuration"],
        "urgency_level": "medium",
        "contains_sarcasm": False,
        "language": "en",
        "expected_intent": "technical_question",
        "has_attachments": False,
        "os": "Linux (Ubuntu 22.04)",
        "additional_info": None
    },
    {
        "id": "req005",
        "customer_name": "Marga Milch",
        "text": "Your documentation says the feature is available, but I can't find it anywhere in the dashboard. Great job on the hide and seek implementation.",
        "product_type": "saas_platform",
        "categories": ["documentation", "ui", "feature_access"],
        "urgency_level": "low",
        "contains_sarcasm": True,
        "language": "en",
        "expected_intent": "feature_request",
        "has_attachments": False,
        "os": None,
        "additional_info": None
    }
]

(DATA_DIR / "support_requests.json").write_text(json.dumps(support_requests, indent=2), encoding="utf-8")
print(f"Wrote {len(support_requests)} support requests.")

Wrote 5 support requests.


### Step 4: Create few-shot examples for intent detection

In [55]:
examples = [
    
    # Intent classification examples
    {
        "text": "The app crashes when I try to export data.",
        "intent": "bug_report",
        "urgency": "high",
        "reason": "User reports a malfunction blocking workflow."
    },
    {
        "text": "How do I reset my password?",
        "intent": "technical_question",
        "urgency": "medium",
        "reason": "User asks for help with a feature."
    },
    {
        "text": "I need to add more users to my account.",
        "intent": "feature_request",
        "urgency": "low",
        "reason": "User wants additional functionality."
    },
    {
        "text": "Great, another bug in your update.",
        "intent": "bug_report",
        "urgency": "critical",
        "reason": "Sarcastic remark about a problem preventing work."
    },
    {
        "text": "Why am I being charged for unused features?",
        "intent": "billing",
        "urgency": "medium",
        "reason": "User questions charges."
    },

    # Urgency classification examples
    {
        "text": "The entire system is down and we cannot process any orders!",
        "urgency": "critical",
        "intent": "bug_report",
        "reason": "System outage affects core business operations."
    },
    {
        "text": "I get an error when trying to save large files.",
        "urgency": "high",
        "intent": "bug_report",
        "reason": "Error blocks important functionality."
    },
    {
        "text": "Where can I find the API documentation?",
        "urgency": "low",
        "intent": "technical_question",
        "reason": "General information request with no time pressure."
    }
]

(DATA_DIR / "few_shot_examples.json").write_text(json.dumps(examples, indent=2), encoding="utf-8")
print(f"Wrote {len(examples)} few-shot examples.")

Wrote 8 few-shot examples.


### Step 5: Create task configurations

In [56]:
task_configs = {
    "project": {
        "name": "Customer Support Intent Detection",
        "description": "Jinja templates for dynamic intent detection prompts in customer support.",
        "default_language": "English"
    },
    "tasks": [
        {
            "name": "intent_classification",
            "mode": "document",
            "labels": ["bug_report", "technical_question", "feature_request", "billing", "feedback"],
            "allow_mixed": False,
            "include_few_shot": True,
            "include_reasoning_field": True,
            "json_only": True,
            "max_examples": 3
        },
        {
            "name": "urgency_classification",
            "mode": "document",
            "labels": ["critical", "high", "medium", "low"],
            "allow_mixed": False,
            "include_few_shot": True,
            "include_reasoning_field": True,
            "json_only": True,
            "max_examples": 3
        },
        {
            "name": "category_based",
            "mode": "category",
            "labels": ["bug_report", "technical_question", "feature_request", "billing", "feedback", "not_applicable"],
            "allow_mixed": False,
            "include_few_shot": False,
            "include_reasoning_field": False,
            "json_only": True,
            "max_examples": 0
        }
    ]
}

(DATA_DIR / "task_configs.json").write_text(json.dumps(task_configs, indent=2), encoding="utf-8")
print(f"Wrote {len(task_configs['tasks'])} task configurations.")

Wrote 3 task configurations.


### Step 6: Define Jinja Templates

The `templates` directory contains three jinja templates (and one macro template) that are based on the ones defined in the sentiment demo discussed during the exercise:

- `intent_prompt.j2`: Single support request containing logic to adjust the prompt based on the mode, included sarcasm, ...
- `batch_prompt.j2`: Multiple support requests using the for-loop logic
- `report.md.j2`: For summarizing the dataset and project in a Markdown report. This also contains the explanation why Jinja is useful here
- (`macros.j2`: Macros for list formatting)

### Step 7: Render the request intent detection prompts

In [57]:
# Define the environment
env = Environment(
    loader=FileSystemLoader(TEMPLATES_DIR),
    undefined=StrictUndefined,
    trim_blocks=True,
    lstrip_blocks=True,
)

# Get the temlates
single_prompt = env.get_template("intent_prompt.txt.j2")
batch_prompt = env.get_template("batch_prompt.txt.j2")
report = env.get_template("report.md.j2")

# Get the project and tasks
project = task_configs["project"]
tasks = task_configs["tasks"]

# Loop over the tasks
for task in tasks:

    # Define the output dir
    task_output_dir = OUTPUTS_DIR / task["name"]
    task_output_dir.mkdir(parents=True, exist_ok=True)

    # Render all the requests using the single request template
    for support_req in support_requests:
        rendered = single_prompt.render(
            project=project,
            task=task,
            request=support_req,
            examples=examples,
        )

        # Add the rendered template to the output dir
        (task_output_dir / f"{support_req['id']}.txt").write_text(rendered, encoding="utf-8")

    # Render all the support requests using the batch_prompt template and save the result 
    rendered_batch = batch_prompt.render(task=task, requests=support_requests)
    (task_output_dir / "batch_prompt.txt").write_text(rendered_batch, encoding="utf-8")

# Get the ground truth
gold_distribution = dict(Counter(support_req["expected_intent"] for support_req in support_requests))

# Render the report and save it as well
rendered_report = report.render(
    project=project,
    requests=support_requests,
    tasks=tasks,
    gold_distribution=gold_distribution,
)
(OUTPUTS_DIR / "project_report.md").write_text(rendered_report, encoding="utf-8")


# List the generated files
print("Generated files:")
for path in sorted(OUTPUTS_DIR.rglob("*")):
    if path.is_file():
        print("-", path)


Generated files:
- outputs/category_based/batch_prompt.txt
- outputs/category_based/req001.txt
- outputs/category_based/req002.txt
- outputs/category_based/req003.txt
- outputs/category_based/req004.txt
- outputs/category_based/req005.txt
- outputs/intent_classification/batch_prompt.txt
- outputs/intent_classification/req001.txt
- outputs/intent_classification/req002.txt
- outputs/intent_classification/req003.txt
- outputs/intent_classification/req004.txt
- outputs/intent_classification/req005.txt
- outputs/project_report.md
- outputs/urgency_classification/batch_prompt.txt
- outputs/urgency_classification/req001.txt
- outputs/urgency_classification/req002.txt
- outputs/urgency_classification/req003.txt
- outputs/urgency_classification/req004.txt
- outputs/urgency_classification/req005.txt


### Step 8: Inspect some Prompts

#### Intent Classification

In [58]:
print((OUTPUTS_DIR / "intent_classification" / "req002.txt").read_text(encoding="utf-8"))

You are an expert customer support classifier.

Project: Customer Support Intent Detection
Task variant: intent_classification
Task mode: document
Request ID: req002
Customer: Rainer Zufall
Product: mobile_app
Language: en
Urgency: critical

Text:
"Oh fantastic, the 'Improve User Experience' update broke the entire user experience. Now nothing works."

Allowed intent labels: `bug_report`, `technical_question`, `feature_request`, `billing`, `feedback`

Classify the primary intent of this support request.

Important: this request may contain sarcasm. Interpret the intended meaning, not only the literal wording.
Examples:
Input: "The app crashes when I try to export data."
Output:
{
"intent": "bug_report",
  "reason": "User reports a malfunction blocking workflow."
}
Input: "How do I reset my password?"
Output:
{
"intent": "technical_question",
  "reason": "User asks for help with a feature."
}
Input: "I need to add more users to my account."
Output:
{
"intent": "feature_request",
  "reas

#### Urgency Classification

In [59]:
print((OUTPUTS_DIR / "urgency_classification" / "req001.txt").read_text(encoding="utf-8"))

You are an expert customer support classifier.

Project: Customer Support Intent Detection
Task variant: urgency_classification
Task mode: document
Request ID: req001
Customer: Max Mustermann
Product: data_analytics_tool
Language: en

Text:
"The application crashes every time I try to export CSV files larger than 10MB. I've attached the error log."

Allowed urgency labels: `critical`, `high`, `medium`, `low`

Classify the urgency level of this support request based solely on the text content and context. Do NOT use any metadata that might be available in the system.


Note: The customer has attached error logs or additional files.
Examples:
Input: "The app crashes when I try to export data."
Output:
{
"urgency": "high",
  "reason": "User reports a malfunction blocking workflow."
}
Input: "How do I reset my password?"
Output:
{
"urgency": "medium",
  "reason": "User asks for help with a feature."
}
Input: "I need to add more users to my account."
Output:
{
"urgency": "low",
  "reason": 

#### Category based

In [60]:
print((OUTPUTS_DIR / "category_based" / "req003.txt").read_text(encoding="utf-8"))

You are an expert customer support classifier.

Project: Customer Support Intent Detection
Task variant: category_based
Task mode: category
Request ID: req003
Customer: Ernst Haft
Product: api_service
Language: en
Urgency: high

Text:
"I need help with API authentication and also with setting up webhook notifications for our enterprise plan."

Allowed intent labels: `bug_report`, `technical_question`, `feature_request`, `billing`, `feedback`, `not_applicable`

Classify intent separately for each category:
- api
- authentication
- webhooks
- enterprise
Use `not_applicable` only if the request does not contain enough information about that category.

Return only valid JSON.

Required JSON schema:
{
  "request_id": "req003",
  "category_intents": [
{
      "category": "api",
      "intent": "one of: bug_report, technical_question, feature_request, billing, feedback, not_applicable",
      "evidence": "short phrase from the request supporting this intent for the category"
    },{
      "ca

### Step 9: Inspect a Report
Render a report based on the code of the sentiment demo discussed in the exercise.

In [61]:
display(Markdown((OUTPUTS_DIR / "project_report.md").read_text(encoding="utf-8")))

# Customer Support Intent Detection

Jinja templates for dynamic intent detection prompts in customer support.

## Dataset Summary

Total support requests: 5

Requests with possible sarcasm: 2

Requests with attachments: 1

Urgency distribution:
- high: 2
- critical: 1
- medium: 1
- low: 1

Intent distribution:
- req001: bug_report
- req002: bug_report
- req003: technical_support
- req004: technical_question
- req005: feature_request

## Prompt Variants

### 1. intent_classification

- Mode: `document`
- Labels: bug_report, technical_question, feature_request, billing, feedback
- Few-shot examples: yes, up to 3- JSON-only response: yes
This variant performs standard intent classification.

### 2. urgency_classification

- Mode: `document`
- Labels: critical, high, medium, low
- Few-shot examples: yes, up to 3- JSON-only response: yes
This variant classifies the urgency level of support requests instead of their intent, demonstrating how the same template structure can handle completely different classification tasks.

### 3. category_based

- Mode: `category`
- Labels: bug_report, technical_question, feature_request, billing, feedback, not_applicable
- Few-shot examples: no- JSON-only response: yes
This variant changes the output schema from one document-level label to a list of category-level intents.



### Step 10: Why Jinja Is Useful Here

This project demonstrates why a templating engine like Jinja is superior to plain Python string formatting for NLP prompt generation:

**1. Multiple Task Variants from One Template**

The same `intent_prompt.txt.j2` template generates three completely different prompt types:
- `intent_classification`: Classifies support request intent (bug_report, technical_question, etc.)
- `urgency_classification`: Classifies urgency level (critical, high, medium, low) using the *same template structure*
- `category_based`: Classifies intent separately for each category

Without Jinja, you would need three separate hard-coded Python strings with duplicated logic.

**2. Conditional Logic**

The templates use `{% if task.name == "urgency_classification" %}` to change behavior dynamically:
- Hide urgency metadata when classifying urgency (to avoid data leakage)
- Switch between intent/urgency labels and output schemas
- Show different instructions based on task mode

**3. Loops for Reusable Structures**

`{% for category in request.categories %}` loops over categories for aspect-based classification,
`{% for example in examples[:task.max_examples] %}` handles variable numbers of few-shot examples.

**4. Macros for DRY Principles**

The `comma_list` macro in `macros.j2` formats label lists consistently across all templates.

**5. Different Label Spaces and Output Formats**

- Intent tasks use: `[bug_report, technical_question, feature_request, billing, feedback]`
- Urgency tasks use: `[critical, high, medium, low]`
- Category tasks generate nested JSON arrays

All controlled through `task_configs.json` without touching the template code.

**6. Maintainability**

Changing a prompt means editing the template file, not Python code. Adding a new task variant
only requires adding a configuration entry, not rewriting string logic.
